# Notebook 3: Evaluation (CodeBLEU + Exact Match) and RAG Comparison

## Evaluates all four configurations on the same held-out test set:
   1. Pipeline A — T5-small fine-tuned WITH pre-training
   2. Pipeline B — T5-small fine-tuned WITHOUT pre-training
   3. Qwen 2.5-Coder-1.5B — Zero-shot (no retrieval)
   4. Qwen 2.5-Coder-1.5B — RAG (≥3-shot with retrieved examples)

## RAG:
- Retriever: CodeBERT (microsoft/codebert-base) encoder with mean-pooling + FAISS
- Knowledge base: bug-fixing training pairs (buggy -> fixed)

In [ ]:
#!pip install datasets transformers tokenizers torch faiss-cpu evaluate sacrebleu evaluate tree-sitter==0.21.0 tree-sitter-java==0.20.0 codebleu==0.1.7 --quiet

In [8]:
import json
import numpy as np
import re
import torch
from collections import Counter
import math
import faiss
from datasets import load_dataset
from transformers import (
    PreTrainedTokenizerFast,
    T5ForConditionalGeneration,
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModel,
)
from codebleu import calc_codebleu
from tqdm import tqdm
import os

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")


Device: cuda


In [3]:
os.makedirs("./eval_results", exist_ok=True)

print("Directories ready.")

Directories ready.


In [4]:
# Load Dataset and Tokenizer

print("Loading dataset and tokenizer...")
dataset = load_dataset("google/code_x_glue_cc_code_refinement", name="medium")

# Same tokenizer used throughout
tokenizer = PreTrainedTokenizerFast.from_pretrained("./java_tokenizer")
print(f"Tokenizer vocab size: {len(tokenizer)}")

# Use the full test set and evaluate only once, for final reporting
test_data = dataset["test"]
print(f"Test samples: {len(test_data)}")

print("\n Test Example [0]...")
print("Buggy :", test_data[0]["buggy"][:120])
print("Fixed :", test_data[0]["fixed"][:120])

Loading dataset and tokenizer...
Tokenizer vocab size: 16384
Test samples: 6545

 Test Example [0]...
Buggy : public java.lang.String METHOD_1 ( ) { if ( ( METHOD_2 ( ) ) && ( METHOD_3 ( VAR_1 . METHOD_4 ( ) ) ) ) { return VAR_2 .
Fixed : public java.lang.String METHOD_1 ( ) { if ( ( METHOD_2 ( ) ) && ( METHOD_3 ( VAR_1 . METHOD_4 ( ) ) ) ) { return VAR_1 .


In [ ]:
def compute_exact_match(predictions, references):
    """
    Exact match accuracy: fraction of predictions that exactly match reference.
    Strips leading/trailing whitespace before comparison.
    """
    assert len(predictions) == len(references), "Length mismatch"
    matches = sum(p.strip() == r.strip() for p, r in zip(predictions, references))
    return round(matches / len(references), 4)

# compute bleu, try catch for errors
def compute_codebleu_score(predictions, references, lang="java"):
    """
    CodeBLEU: code-aware metric combining n-gram, syntax, and dataflow scores.
    """
    try:
        # Ensure we are passing lists
        if isinstance(predictions, str): predictions = [predictions]
        if isinstance(references, str): references = [references]

        result = calc_codebleu(references, predictions, lang=lang, weights=(0.25, 0.25, 0.25, 0.25))
        return result
    except Exception as e:
        print(f"CodeBLEU Error: {e}")
        return {
            "codebleu": 0.0,
            "ngram_match_score": 0.0,
            "syntax_match_score": 0.0,
            "dataflow_match_score": 0.0
        }

# generate predictions

def generate_t5_predictions(model, tokenizer, test_data, batch_size=16, prefix="fix: "):
    """
    Generate predictions from a T5 model on the test set.
    Returns a list of decoded prediction strings.
    """
    model.eval()
    model.to(DEVICE)
    predictions = []

    for i in tqdm(range(0, len(test_data), batch_size), desc="Generating"):
        batch = test_data[i : i + batch_size]
        inputs = [prefix + b for b in batch["buggy"]]

        encoded = tokenizer(
            inputs,
            return_tensors="pt",
            max_length=512,
            truncation=True,
            padding=True,
        ).to(DEVICE)

        with torch.no_grad():
            output_ids = model.generate(
                input_ids=encoded["input_ids"],
                attention_mask=encoded["attention_mask"],
                max_new_tokens=256,
                num_beams=4,            # Beam search for better output quality
                early_stopping=True,
            )

        decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        predictions.extend(decoded)

    return predictions

# evaluate model

def evaluate_model(predictions, references, label):
    """
    Compute and print CodeBLEU and exact match for a set of predictions.
    Returns a results dict for the summary table.
    """
    em = compute_exact_match(predictions, references)
    cb = compute_codebleu_score(predictions, references, lang="java")

    print(f"\n{'-'*50}")
    print(f"Results: {label}")
    print(f"{'-'*50}")
    print(f"  Exact Match  : {em:.4f}  ({em*100:.2f}%)")
    print(f"  CodeBLEU     : {cb['codebleu']:.4f}")
    print(f"    n-gram     : {cb['ngram_match_score']:.4f}")
    print(f"    syntax     : {cb['syntax_match_score']:.4f}")
    print(f"    dataflow   : {cb['dataflow_match_score']:.4f}")

    return {
        "label": label,
        "exact_match": em,
        "codebleu": cb["codebleu"],
        "ngram_match": cb["ngram_match_score"],
        "syntax_match": cb["syntax_match_score"],
        "dataflow_match": cb["dataflow_match_score"],
    }


references = list(test_data["fixed"])

In [7]:
# Pipeline A eval T5 Fine-tuned with Pre-training

print("\nLoading Pipeline A model (with pre-training)...")
model_a = T5ForConditionalGeneration.from_pretrained("./pipeline_a_finetuned")

preds_a = generate_t5_predictions(model_a, tokenizer, test_data)



Loading Pipeline A model (with pre-training)...


Loading weights:   0%|          | 0/143 [00:00<?, ?it/s]

Generating: 100%|██████████| 410/410 [10:18<00:00,  1.51s/it]


In [ ]:
# output_file = "./eval_results/predA_results.json"

# with open(output_file, "w") as f:
#     json.dump(preds_a, f, indent=4)

# print(f"Saved {len(preds_a)} outputs to {output_file}")

# commented this out because ran out of memory and needed to load back in but all
# saved in the final results

Saved 6545 outputs to ./eval_results/predA_results.json


In [ ]:
# results from the pipeline A 

results_a = evaluate_model(preds_a, references, "Pipeline A — With Pre-training")

# Free GPU memory before loading next model
del model_a
torch.cuda.empty_cache()


--------------------------------------------------
Results: Pipeline A — With Pre-training
--------------------------------------------------
  Exact Match  : 0.0003  (0.03%)
  CodeBLEU     : 0.6658
    n-gram     : 0.4558
    syntax     : 0.7000
    dataflow   : 0.7573


### Pipeline B - Without Pretraining

In [6]:
# Pipeline B Eval T5 Fine-tuned without Pre-training
print("\nLoading Pipeline B model (without pre-training)...")
model_b = T5ForConditionalGeneration.from_pretrained("./pipeline_b_finetuned")

preds_b = generate_t5_predictions(model_b, tokenizer, test_data)



Loading Pipeline B model (without pre-training)...


Loading weights:   0%|          | 0/143 [00:00<?, ?it/s]

Generating: 100%|██████████████████████████████████████████████████████████████████████████████████| 410/410 [10:20<00:00,  1.51s/it]


In [ ]:
results_b = evaluate_model(preds_b, references, "Pipeline B — Without Pre-training")

# Free gpu memory 
del model_b
torch.cuda.empty_cache()


--------------------------------------------------
Results: Pipeline B — Without Pre-training
--------------------------------------------------
  Exact Match  : 0.0006  (0.06%)
  CodeBLEU     : 0.7311
    n-gram     : 0.7293
    syntax     : 0.7125
    dataflow   : 0.7180


## SECTION B: RAG System — CodeBERT Retriever + Qwen 2.5-Coder-1.5B Generator

# Retriever design choice:
-   CodeBERT (microsoft/codebert-base) with mean-pooling over the last hidden state.
-  CodeBERT was pre-trained on code and understands code semantics better than general-purpose sentence embedders.
-  Mean-pooling produces a single fixed-size vector per method — suitable for FAISS similarity search.

### Alternative considered: Code2Vec (as in the RAG lab) — rejected because Code2Vec requires AST parsing and is Java-specific. CodeBERT is more general and better captures semantic similarity for this retrieval task.

In [18]:
print("\nLoading CodeBERT for retrieval...")
codebert_tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
codebert_model = AutoModel.from_pretrained("microsoft/codebert-base").to(DEVICE)
codebert_model.eval()
print("\n Finished Load")
 

def encode_with_codebert(code_list, batch_size=64):
    """
    Encode a list of code strings using CodeBERT with mean-pooling.
    Returns a numpy array of shape (len(code_list), 768).
    """
    all_embeddings = []
    for i in tqdm(range(0, len(code_list), batch_size), desc="Encoding"):
        batch = code_list[i : i + batch_size]
        encoded = codebert_tokenizer(
            batch,
            return_tensors="pt",
            max_length=512,
            truncation=True,
            padding=True,
        ).to(DEVICE)

        with torch.no_grad():
            outputs = codebert_model(**encoded)

        # Mean-pool over token dimension, ignoring padding tokens
        attention_mask = encoded["attention_mask"].unsqueeze(-1).float()
        token_embeddings = outputs.last_hidden_state
        mean_embeddings = (token_embeddings * attention_mask).sum(1) / attention_mask.sum(1)
        all_embeddings.append(mean_embeddings.cpu().numpy())

    return np.vstack(all_embeddings)


Loading CodeBERT for retrieval...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


 Finished Load


In [19]:
print("\nBuilding FAISS index from training knowledge base...")
train_buggy = list(dataset["train"]["buggy"])
train_fixed = list(dataset["train"]["fixed"])
 
kb_embeddings = encode_with_codebert(train_buggy)
 
dimension = kb_embeddings.shape[1]  # 768 for CodeBERT
index = faiss.IndexFlatL2(dimension)
index.add(kb_embeddings.astype(np.float32))
print(f"FAISS index built | Vectors: {index.ntotal} | Dimension: {dimension}")


Building FAISS index from training knowledge base...


Encoding: 100%|████████████████████████████████████████████████████████████████████████████████████| 819/819 [02:53<00:00,  4.73it/s]

FAISS index built | Vectors: 52364 | Dimension: 768


In [20]:
# Pre-encode all test queries once instead of re-encoding per sample,
# encode everything upfront and store results.
 
print("\nPre-encoding all test queries with CodeBERT...")
test_buggy_list = [d["buggy"] for d in test_data]
test_embeddings = encode_with_codebert(test_buggy_list)  # shape: (N_test, 768)
print(f"Test queries encoded: {test_embeddings.shape}")
 
# Free CodeBERT from GPU after all encoding is done 
del codebert_model
torch.cuda.empty_cache()
print("CodeBERT freed from GPU.")


Pre-encoding all test queries with CodeBERT...


Encoding: 100%|████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:21<00:00,  4.77it/s]

Test queries encoded: (6545, 768)
CodeBERT freed from GPU.


In [ ]:
# Retrieval functions 
 
def retrieve_examples_batch(query_embeddings, k=3):
    """
    Retrieve k most similar bug-fix examples for a BATCH of queries at once.
    query_embeddings: numpy array of shape (batch_size, 768)
    Returns list of lists of (buggy, fixed) tuples.
    """
    _, indices = index.search(query_embeddings.astype(np.float32), k)
    return [
        [(train_buggy[int(idx)], train_fixed[int(idx)]) for idx in row]
        for row in indices
    ]
 

def retrieve_examples_single(query_idx, k=3):
    """
    Retrieve k examples for a single test sample using its pre-computed embedding.
    """
    query_emb = test_embeddings[query_idx : query_idx + 1]
    _, indices = index.search(query_emb.astype(np.float32), k)
    return [(train_buggy[int(i)], train_fixed[int(i)]) for i in indices[0]]
 

def build_rag_prompt(buggy_code, query_idx, k=3):
    """
    Build a 3-shot prompt using retrieved bug-fix examples.
    Uses pre-computed embedding via query_idx
    """
    examples = retrieve_examples_single(query_idx, k=k)
    prompt = (
        "You are an expert Java developer. Fix the bug in the Java method below.\n"
        "Here are similar bug-fix examples for reference:\n\n"
    )
    for idx, (bug, fix) in enumerate(examples, 1):
        prompt += f"### Example {idx}\nBuggy:\n{bug}\n\nFixed:\n{fix}\n\n"
 
    prompt += f"### Now fix this method\nBuggy:\n{buggy_code}\n\nFixed:\n"
    return prompt
 

    
def build_zeroshot_prompt(buggy_code, query_idx=None):
    """
    Zero-shot prompt — no retrieved examples, just the buggy method.
    query_idx param accepted but unused (keeps signature uniform).
    """
    return (
        "You are an expert Java developer. Fix the bug in the following Java method.\n"
        f"Buggy:\n{buggy_code}\n\nFixed:\n"
    )
 

In [22]:
# Quick retrieval sanity check 
 
print("\nRetrieval Check (using pre-encoded embeddings)...")
sample_bug = test_data[0]["buggy"]
retrieved = retrieve_examples_single(0, k=3)
print(f"Query : {sample_bug[:80]}...")
for i, (b, f) in enumerate(retrieved, 1):
    print(f"  Retrieved {i}: {b[:80]}...")


Retrieval Check (using pre-encoded embeddings)...
Query : public java.lang.String METHOD_1 ( ) { if ( ( METHOD_2 ( ) ) && ( METHOD_3 ( VAR...
  Retrieved 1: private boolean METHOD_1 ( ) { VAR_1 = null ; if ( VAR_2 . METHOD_2 ( ( ( VAR_3 ...
  Retrieved 2: public TYPE_1 METHOD_1 ( ) { if ( ( this . METHOD_2 ( ) ) || ( this . METHOD_3 (...
  Retrieved 3: public java.lang.Void METHOD_1 ( ) { if ( ! ( VAR_1 . METHOD_2 ( ) . METHOD_3 ( ...


In [23]:
# Load Qwen 
 
print("\nLoading Qwen2.5-Coder-1.5B-Instruct...")
QWEN_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
 
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL, trust_remote_code=True)
 
# Set padding side to left for causal LM batched generation.
# sequences in a batch
qwen_tokenizer.padding_side = "left"
if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)
qwen_model.eval()
print("Qwen model loaded.")
 


Loading Qwen2.5-Coder-1.5B-Instruct...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen model loaded.


In [ ]:
# Batched generation 
 
def generate_qwen_predictions(test_data, prompt_fn, label, batch_size=8):
    """
    Generate predictions from Qwen using the given prompt builder function.
    - prompt_fn receives (buggy_code, query_idx) so RAG can use
      pre-computed embeddings without any model reload
    - Left-padded tokenizer ensures correct generation for all batch items
    - Prompts are built for the whole batch before tokenizing together
 
    Returns list of predicted fixed methods.
    """
    predictions = []
    n = len(test_data)
 
    for batch_start in tqdm(range(0, n, batch_size), desc=f"Qwen ({label})"):
        batch_indices = range(batch_start, min(batch_start + batch_size, n))
 
        # Build all prompts for this batch
        prompts = [
            prompt_fn(test_data[i]["buggy"], i)
            for i in batch_indices
        ]
 
        # Tokenize as a left-padded batch
        inputs = qwen_tokenizer(
            prompts,
            return_tensors="pt",
            max_length=1024,
            truncation=True,
            padding=True,          # pads to longest in batch (left side)
        ).to(DEVICE)
 
        with torch.no_grad():
            output_ids = qwen_model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,   # greedy — deterministic
                pad_token_id=qwen_tokenizer.eos_token_id,
            )
 
        # Decode only newly generated tokens for each item in the batch
        input_len = inputs["input_ids"].shape[1]
        for seq in output_ids:
            new_tokens = seq[input_len:]
            decoded = qwen_tokenizer.decode(new_tokens, skip_special_tokens=True)
            prediction = decoded.strip().split("\n\n")[0].strip()
            predictions.append(prediction)
 
    return predictions
 

In [25]:
# Zero-Shot Evaluation 
 
print("\nRunning Qwen Zero-Shot...")
preds_zeroshot = generate_qwen_predictions(
    test_data,
    build_zeroshot_prompt,
    "zero-shot",
    batch_size=4,  
)
results_zeroshot = evaluate_model(preds_zeroshot, references, "Qwen 1.5B — Zero-Shot")


Running Qwen Zero-Shot...


Qwen (zero-shot): 100%|████████████████████████████████████████████████████████████████████████| 1637/1637 [1:23:49<00:00,  3.07s/it]



--------------------------------------------------
Results: Qwen 1.5B — Zero-Shot
--------------------------------------------------
  Exact Match  : 0.0000  (0.00%)
  CodeBLEU     : 0.3574
    n-gram     : 0.0678
    syntax     : 0.5583
    dataflow   : 0.6857


In [26]:
print("\nRunning Qwen RAG (3-shot)...")
preds_rag = generate_qwen_predictions(
    test_data,
    build_rag_prompt,
    "RAG-3shot",
    batch_size=4,   
)
results_rag = evaluate_model(preds_rag, references, "Qwen 1.5B — RAG 3-shot")


Running Qwen RAG (3-shot)...


Qwen (RAG-3shot): 100%|████████████████████████████████████████████████████████████████████████| 1637/1637 [1:40:59<00:00,  3.70s/it]



--------------------------------------------------
Results: Qwen 1.5B — RAG 3-shot
--------------------------------------------------
  Exact Match  : 0.0000  (0.00%)
  CodeBLEU     : 0.5409
    n-gram     : 0.4283
    syntax     : 0.6228
    dataflow   : 0.6675


In [30]:
# Final Results Summary Table

all_results = [results_a, results_b, results_zeroshot, results_rag]

print("\n" + "-"*70)
print("FINAL RESULTS — All Four Configurations")
print("-"*70)
print(f"{'Configuration':<40} {'Exact Match':>12} {'CodeBLEU':>10}")
print("-"*70)
for r in all_results:
    print(f"{r['label']:<40} {r['exact_match']:>11.4f}  {r['codebleu']:>9.4f}")
print("-"*70)

print("\nCodeBLEU breakdown:")
print(f"{'Configuration':<40} {'n-gram':>8} {'syntax':>8} {'dataflow':>10}")
print("-"*70)
for r in all_results:
    print(
        f"{r['label']:<40} {r['ngram_match']:>8.4f} "
        f"{r['syntax_match']:>8.4f} {r['dataflow_match']:>10.4f}"
    )

# Save results to JSON for the report
with open("./eval_results/results_summary.json", "w") as f:
    json.dump(all_results, f, indent=2)
print("\nResults saved to ./eval_results/results_summary.json")



----------------------------------------------------------------------
FINAL RESULTS — All Four Configurations
----------------------------------------------------------------------
Configuration                             Exact Match   CodeBLEU
----------------------------------------------------------------------
Pipeline A — With Pre-training                0.0003     0.6658
Pipeline B — Without Pre-training             0.0006     0.7311
Qwen 1.5B — Zero-Shot                         0.0000     0.3574
Qwen 1.5B — RAG 3-shot                        0.0000     0.5409
----------------------------------------------------------------------

CodeBLEU breakdown:
Configuration                              n-gram   syntax   dataflow
----------------------------------------------------------------------
Pipeline A — With Pre-training             0.4558   0.7000     0.7573
Pipeline B — Without Pre-training          0.7293   0.7125     0.7180
Qwen 1.5B — Zero-Shot                      0.0678   

Output File Directories:

Results for both Pipelines and RAG
- `./eval_results/results_summary.json`

## Conclusion + Key Findings

These notebooks compared three paradigms for automated Java bug fixing at small model scale: pre-train then fine-tune (Pipeline A), fine-tuning from scratch (Pipeline B), and retrieval-augmented prompting (RAG with Qwen 1.5B)

### key Findings:
- Pipeline B (from scratch) recieved CodeBLEU 0.7311, and outperformed Pipeline A (CodeBLEU 0.6658), suggesting that at this corpus scale of ~50k methods and 3 epochs, pretraining did not provide a useful initialization advantage for the dowstream bug-fixing task.

- Both fine-tuned T5-small models outperformed Qwen 1.5B with RAG (CodeBLEU 0.5409), demonstrating that task-specific supervised training is more reliable for size-driven general capability when sufficient labeled data is available. 

- RAG provided a great boost over zero-shot prompting (0.3574 to 0.5409), validating the CodeBERT retriever design and confirming that semantic retrieval effectively uncovers relavent bug-fix patterns.